In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 8 Lab — Unsupervised Learning: Clustering
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Exploring a dataset with no target variable
- Scaling features and understanding why it is non-optional for clustering
- Choosing `k` with the elbow method and silhouette score
- Fitting KMeans and profiling the resulting clusters
- Fitting hierarchical clustering and reading a dendrogram

**Estimated time:** 85–100 minutes

---

## The Dataset

This lab uses a synthetic customer behavioural dataset. Each row is one customer. There is no target variable — no column that says what segment each customer belongs to. Your job is to find the natural groupings in the data and give them business meaning.

The features are modelled on **RFM analysis** — Recency, Frequency, and Monetary value — a standard framework for customer segmentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(5)
n = 350

# Three latent segments — NOT included in the DataFrame (students discover them)
seg = np.random.choice(['loyalist', 'casual', 'at_risk'], n, p=[0.25, 0.45, 0.30])

recency_days = np.where(seg == 'loyalist', np.random.randint(1, 30, n),
               np.where(seg == 'casual',   np.random.randint(15, 90, n),
                                           np.random.randint(60, 365, n)))

frequency    = np.where(seg == 'loyalist', np.random.randint(12, 36, n),
               np.where(seg == 'casual',   np.random.randint(3, 12, n),
                                           np.random.randint(1, 5, n)))

avg_order    = np.where(seg == 'loyalist',
                   np.round(np.random.uniform(120, 300, n), 2),
               np.where(seg == 'casual',
                   np.round(np.random.uniform(40, 120, n), 2),
                   np.round(np.random.uniform(20, 80, n), 2)))

num_cat      = np.where(seg == 'loyalist', np.random.randint(5, 10, n),
               np.where(seg == 'casual',   np.random.randint(2, 6, n),
                                           np.random.randint(1, 4, n)))

loyalty_card = np.where(seg == 'loyalist',
                   np.random.choice([0, 1], n, p=[0.10, 0.90]),
               np.where(seg == 'casual',
                   np.random.choice([0, 1], n, p=[0.60, 0.40]),
                   np.random.choice([0, 1], n, p=[0.80, 0.20])))

region       = np.random.choice(['North', 'South', 'East', 'West'], n)

df = pd.DataFrame({
    'recency_days':   recency_days,
    'frequency':      frequency,
    'avg_order_value': avg_order,
    'num_categories': num_cat,
    'loyalty_card':   loyalty_card,
    'region':         region,
})

print(f'Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns')
print(df.dtypes)
print()
print(df.describe().round(1))

---
## Part 1 — Explore the Data and Understand Scaling

Before clustering, understand the feature ranges. KMeans uses Euclidean distance — features on a large scale dominate the distance calculation and make smaller-scale features invisible to the model.

In [ ]:
# Feature ranges
print('Feature ranges:')
for col in ['recency_days', 'frequency', 'avg_order_value', 'num_categories', 'loyalty_card']:
    print(f'  {col:<20} min={df[col].min():>6.1f}  max={df[col].max():>6.1f}')

# Correlation matrix — which features move together?
num_cols = ['recency_days', 'frequency', 'avg_order_value', 'num_categories', 'loyalty_card']
corr = df[num_cols].corr().round(2)
print('\nCorrelation matrix:')
print(corr)

---
### 🤖 ML Connection — Why Scaling Is Non-Optional for KMeans

`avg_order_value` ranges from ~$20 to ~$300. `num_categories` ranges from 1 to 9. Without scaling, a difference of $50 in order value and a difference of 5 in categories are not comparable — the dollar difference will dominate every distance calculation simply because $50 >> 5. After scaling both to mean 0 and std 1, a one-unit change in each feature represents the same relative departure from the average. Every feature contributes equally to the distance calculation.

---

### ✏️ Exercise 1.1 — Scale and Visualise

1. Scale the five numeric features using `StandardScaler`. Store the result as `X_scaled` and the feature names as `feature_cols`.
2. Print the mean and standard deviation of each scaled column to confirm scaling worked.
3. Create a `px.scatter` chart of `recency_days` vs. `avg_order_value` (unscaled) to visualise the raw distribution. Does any natural grouping appear visually?
4. Write a comment: which single feature has the largest raw range? What would happen to cluster assignments if you forgot to scale it?

In [ ]:
# Exercise 1.1

feature_cols = ['recency_days', 'frequency', 'avg_order_value', 'num_categories', 'loyalty_card']

# 1. Scale
scaler   = StandardScaler()
X_scaled = 

# 2. Confirm scaling
scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print('Scaled feature stats (expect mean≈0, std≈1):')
print(scaled_df.describe().loc[['mean', 'std']].round(3))

# 3. Scatter — recency vs. avg_order_value (unscaled)
fig = px.scatter(
    df,
    x='recency_days', y='avg_order_value',
    color='loyalty_card',
    title='Recency vs. Average Order Value (unscaled)',
    labels={'recency_days': 'Days Since Last Purchase',
            'avg_order_value': 'Average Order Value ($)',
            'loyalty_card': 'Has Loyalty Card'},
    opacity=0.6
)
fig.show()

# 4. Largest raw range and scaling consequence
# 

---
## Part 2 — Choosing k with Elbow and Silhouette

KMeans requires you to specify the number of clusters before fitting. The elbow method and silhouette score give quantitative guidance — but the final choice also depends on whether the resulting clusters are interpretable as business segments.

In [ ]:
# Elbow method — inertia for k = 2 to 9
k_range   = range(2, 10)
inertias  = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_range), inertias, marker='o', color='#4575b4')
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(list(k_range), sil_scores, marker='o', color='#d73027')
axes[1].set_xlabel('k (number of clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k')

plt.tight_layout()
plt.show()

print('k  |  Inertia  |  Silhouette')
for k, ine, sil in zip(k_range, inertias, sil_scores):
    print(f'{k}  |  {ine:>7.1f}  |  {sil:.4f}')

---
### 💼 Business Context — The Business Case Breaks Ties

When the elbow and silhouette methods suggest different values of k, ask: how many distinct customer groups can the business act on differently? If marketing has three campaigns (loyalty programme, win-back, re-engagement) then three segments is the natural ceiling regardless of what the charts suggest. If senior leadership wants a single 'high value' flag, two segments may be more useful than four. Mathematical methods identify candidate values of k; the business use case makes the final call.

---

### ✏️ Exercise 2.1 — Choose and Justify k

1. From the elbow plot, identify where the curve bends. State your candidate k from the elbow method.
2. From the silhouette plot, identify which k has the highest score. Does it agree with the elbow?
3. Choose a final k and write a 2–3 sentence justification that addresses both the quantitative evidence and a business rationale.
4. Print the silhouette scores for k = 2, 3, and 4 side by side.

In [ ]:
# Exercise 2.1

# 1. Elbow candidate k
# 

# 2. Silhouette candidate k
# 

# 3. Final k choice and justification
chosen_k = 
# Justification:
# 
# 

# 4. Silhouette scores for k = 2, 3, 4
for k in [2, 3, 4]:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, lbl)
    print(f'k={k}  silhouette={sil:.4f}')

---
## Part 3 — Fit KMeans and Profile the Clusters

Fitting KMeans with the chosen k produces a cluster label for every customer. Profiling translates those labels into business descriptions.

In [ ]:
# Fit KMeans with chosen k
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

print(f'Cluster sizes:')
print(df['cluster'].value_counts().sort_index())

# Profile — mean of each feature per cluster (unscaled, for interpretability)
profile = df.groupby('cluster')[feature_cols].mean().round(2)
print('\nCluster profiles (unscaled means):')
print(profile)

---
### 🤖 ML Connection — Profile in Original Units, Cluster in Scaled Units

The clustering ran on `X_scaled` — distances were measured in standard deviation units. But the profile table uses `df` (unscaled) because stakeholders understand dollars, days, and counts. A cluster profile that says 'mean recency = 0.72' is meaningless; 'mean recency = 8 days' is immediately interpretable. Scale for the algorithm; unscale for the human.

---

### ✏️ Exercise 3.1 — Name the Segments and Validate

1. Read the cluster profile table. For each cluster, write one sentence describing the typical customer in plain language (e.g., 'Cluster 0 customers purchased recently, buy frequently, and spend the most per order').
2. Assign a business name to each cluster (e.g., 'High-Value Loyalists', 'Occasional Buyers', 'Lapsed Customers'). Add a `segment_name` column to `df`.
3. Check the loyalty card rate per cluster. Does it align with the other features — do higher-value clusters have higher loyalty card adoption?
4. Check whether clusters differ by `region`. Print the region distribution within each cluster. Write a comment: is region a useful differentiator, or do all clusters have similar regional mixes?

In [ ]:
# Exercise 3.1

# 1. Cluster descriptions
# Cluster 0: 
# Cluster 1: 
# Cluster 2: 

# 2. Assign segment names
segment_map  = {
    0: '',   # fill in your name
    1: '',
    2: '',
}
df['segment_name'] = df['cluster'].map(segment_map)
print('Segment name counts:')
print(df['segment_name'].value_counts())

# 3. Loyalty card rate per cluster
loyalty_rate = df.groupby('segment_name')['loyalty_card'].mean().round(3)
print('\nLoyalty card adoption rate by segment:')
print(loyalty_rate)
# Does it align?
# 

# 4. Region distribution by cluster
region_dist = df.groupby(['segment_name', 'region']).size().unstack(fill_value=0)
region_pct  = region_dist.div(region_dist.sum(axis=1), axis=0).round(3)
print('\nRegion distribution by segment (proportions):')
print(region_pct)
# Is region a useful differentiator?
# 

---
## Part 4 — Hierarchical Clustering

Hierarchical clustering builds a nested tree of clusters without requiring you to choose k first. The dendrogram shows where natural breaks exist — you cut at a height that gives a useful number of groups.

In [ ]:
# Compute linkage matrix for the dendrogram
Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(
    Z,
    truncate_mode='lastp',   # show only the last p merges
    p=20,
    show_leaf_counts=True,
    leaf_rotation=45,
)
plt.xlabel('Cluster (n = number of points)')
plt.ylabel('Merge distance (Ward linkage)')
plt.title('Hierarchical Clustering Dendrogram')
plt.tight_layout()
plt.show()

---
### 💼 Business Context — When the Dendrogram Is More Useful Than KMeans

KMeans tells you which cluster each customer belongs to — but not how the clusters relate to each other. The dendrogram shows the merge history: you can see which two sub-groups are most similar, and whether a four-cluster solution is just a three-cluster solution with one group split in two. This is valuable when stakeholders ask: 'If we had to merge two of our four segments into one, which two are most similar?' The dendrogram answers that directly.

---

### ✏️ Exercise 4.1 — Fit and Compare

1. Fit `AgglomerativeClustering` with `n_clusters=3` and `linkage='ward'` on `X_scaled`. Store labels in `df['agg_cluster']`.
2. Build the cluster profile for the agglomerative model and compare it to the KMeans profile. Do the two methods find similar segments?
3. Compute the silhouette score for the agglomerative clusters. Is it higher or lower than KMeans at k=3? What does the difference tell you?
4. Create a cross-tabulation: `pd.crosstab(df['cluster'], df['agg_cluster'])`. Write a comment: do the two clusterings mostly agree on which customers belong together?

In [ ]:
# Exercise 4.1

# 1. Fit agglomerative clustering
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['agg_cluster'] = 

# 2. Profile comparison
agg_profile = df.groupby('agg_cluster')[feature_cols].mean().round(2)
print('KMeans profile:')
print(profile)
print('\nAgglomerative profile:')
print(agg_profile)
# Do they find similar segments?
# 

# 3. Silhouette score comparison
km_sil  = silhouette_score(X_scaled, df['cluster'])
agg_sil = silhouette_score(X_scaled, df['agg_cluster'])
print(f'\nKMeans silhouette (k=3):       {km_sil:.4f}')
print(f'Agglomerative silhouette (k=3): {agg_sil:.4f}')
# 

# 4. Cross-tabulation
crosstab = pd.crosstab(df['cluster'], df['agg_cluster'],
                       rownames=['KMeans'], colnames=['Agglomerative'])
print('\nCross-tabulation:')
print(crosstab)
# Do the clusterings mostly agree?
# 

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

### Visualising Clusters in 2D with PCA

Five features cannot be plotted directly. **Principal Component Analysis (PCA)** compresses multiple features into two dimensions that capture as much variance as possible — a technique covered more formally in advanced courses, but useful here for visualisation.

1. Apply `PCA(n_components=2)` from `sklearn.decomposition` to `X_scaled`. Store the result as `X_pca` with columns `['PC1', 'PC2']`.
2. Add the KMeans cluster labels and segment names to the PCA DataFrame.
3. Create a `px.scatter` chart of `PC1` vs. `PC2`, coloured by `segment_name`. Add hover data showing `recency_days`, `avg_order_value`, and `frequency`.
4. Print the explained variance ratio for each component: `pca.explained_variance_ratio_`. What percentage of total variance is captured by the two components?
5. **Reflection:** If the clusters appear clearly separated in the 2D PCA plot, what does that tell you about the quality of the clustering? If they overlap significantly, what does that suggest?

In [ ]:
# Challenge — PCA visualisation
from sklearn.decomposition import PCA

# 1. Apply PCA
pca   = PCA(n_components=2, random_state=42)
X_pca = 

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['cluster']      = df['cluster'].values
pca_df['segment_name'] = df['segment_name'].values
pca_df['recency_days']    = df['recency_days'].values
pca_df['avg_order_value'] = df['avg_order_value'].values
pca_df['frequency']       = df['frequency'].values

# 3. Scatter plot
fig = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='segment_name',
    hover_data=['recency_days', 'avg_order_value', 'frequency'],
    title='Customer Segments — PCA 2D View',
    opacity=0.7
)
fig.show()

# 4. Explained variance
print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(3)}')
print(f'Total variance captured:  {pca.explained_variance_ratio_.sum():.1%}')

# 5. Reflection
# 

---
## Week 8 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| No target variable | Clustering discovers structure — there is no label, no F1, no ground truth |
| Scale before clustering | KMeans uses Euclidean distance — unscaled features dominate by range |
| `KMeans(n_init=10)` | Run 10 random starts — avoids poor local optima |
| Elbow method | Plot inertia vs. k — look for the bend where gains diminish |
| Silhouette score | Range −1 to 1 — choose k where the score peaks |
| Profile in original units | Scale for the algorithm; unscale for the human |
| Segment naming | Translate numeric profiles into business labels stakeholders can act on |
| Validation features | Check cluster differences on features not used in clustering |
| `AgglomerativeClustering` | Bottom-up merging — no k required upfront |
| Dendrogram | Shows merge history — cut at the largest gap |
| Cross-tabulation | Compare two clusterings — high diagonal counts mean the methods agree |
| PCA (bonus) | Compresses n features to 2D for visualisation |

---
**Next week:** Final Project — you will apply the full workflow from Weeks 1–8 to a dataset of your choice: EDA, wrangling, feature engineering, model selection, evaluation, and stakeholder communication.